# Symposia Getting Started

This notebook keeps one claim and runs it in two modes:

1. Deterministic (`live=False`): reproducible, no provider calls
2. Live (`live=True`): uses provider keys and calls an LLM

If live mode runs, the notebook prints execution evidence (`juror_mode`, model, elapsed time).

In [1]:
from __future__ import annotations
import json
import os
import sys
import time
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

ROOT = Path.cwd()
if ROOT.name == "examples":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from symposia import validate
from symposia.env import load_env
load_env()

CLAIM = (
    "If you always pay your credit card balance in full every month, ",
    "you usually avoid interest charges and keep borrowing costs down.",
)

def summarize_result(result, domain="general"):
    scores = list(result.aggregated_by_subclaim.values())
    escalated = [
        s.subclaim_id
        for s in scores
        if (s.support_score < 0.70) or (s.contradiction_score >= 0.35) or (s.sufficiency_score < 0.70)
    ]
    return {
        "run_id": result.run_id,
        "domain": domain,
        "aggregated_units": len(scores),
        "should_stop": result.completion.should_stop,
        "completion_reason": result.completion.reason,
        "escalated_subclaims": escalated,
    }

def print_summary_block(title: str, summary: dict) -> None:
    print(f"\n{'=' * 56}")
    print(f"{title}")
    print(f"{'-' * 56}")
    print(f"run_id              : {summary['run_id']}")
    print(f"domain              : {summary['domain']}")
    print(f"aggregated_units    : {summary['aggregated_units']}")
    print(f"completion_reason   : {summary['completion_reason']}")
    print(f"should_stop         : {summary['should_stop']}")
    print(f"escalated_subclaims : {summary['escalated_subclaims']}")

## 1) Deterministic Example

No API keys required.

In [2]:
det_result = validate(content="".join(CLAIM), domain="general", live=False)
det_summary = summarize_result(det_result)
print_summary_block("Deterministic Mode", det_summary)


Deterministic Mode
--------------------------------------------------------
run_id              : run_b511ea7135f1
domain              : general
aggregated_units    : 1
completion_reason   : initial_decisive
should_stop         : True
escalated_subclaims : []


/tmp/ipykernel_99006/3867748463.py:1: UserWarning: Symposia is running in deterministic mode (no live provider selected).
  det_result = validate(content="".join(CLAIM), domain="general", live=False)


## 2) Live Example

Requires at least one key in `.env`: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, or `GOOGLE_API_KEY`.

In [3]:
providers = [
    p for p, env in [
        ("openai", "OPENAI_API_KEY"),
        ("anthropic", "ANTHROPIC_API_KEY"),
        ("google", "GOOGLE_API_KEY"),
    ]
    if os.getenv(env)
]

if not providers:
    print("No provider keys detected. Add a key to .env, rerun Cell 2, then rerun this cell.")
else:
    print(f"Detected provider key(s): {providers}")
    t0 = time.perf_counter()
    live_result = validate(
        content="".join(CLAIM) + f" [nonce={time.time_ns()}]",
        domain="general",
        live=True,
    )
    elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)

    votes = live_result.core_trace.juror_votes
    models = sorted({v.provider_model for v in votes if v.provider_model})
    live_summary = summarize_result(live_result)

    print_summary_block("Live Mode", live_summary)
    print(f"execution_mode      : {live_result.execution_policy.get('juror_mode')}")
    print(f"provider_model(s)   : {models if models else 'unknown'}")
    print(f"elapsed_ms          : {elapsed_ms}")
    print(f"runtime_stats       : {live_result.runtime_stats}")

Detected provider key(s): ['openai', 'anthropic', 'google']

Live Mode
--------------------------------------------------------
run_id              : run_26b06e7549c2
domain              : general
aggregated_units    : 1
completion_reason   : initial_decisive
should_stop         : True
escalated_subclaims : []
execution_mode      : llm
provider_model(s)   : ['gpt-4o-mini']
elapsed_ms          : 1617.9
runtime_stats       : {'juror_count': 1, 'total_decisions': 1, 'total_dropouts': 0, 'dropout_rate': 0.0}
